In [9]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu
!pip install numpy matplotlib

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


In [10]:
import torch
import torch.nn as nn
import numpy as np
import random
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

# Set seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Part B: Reproduce the Same RNN in PyTorch

In [12]:
# Reproduce in PyTorch
x1 = torch.tensor([1.0, 0.0, 0.0], dtype=torch.float32)
x2 = torch.tensor([0.0, 1.0, 0.0], dtype=torch.float32)
x3 = torch.tensor([0.0, 0.0, 1.0], dtype=torch.float32)
x4 = torch.tensor([1.0, 1.0, 0.0], dtype=torch.float32)

W_x = torch.tensor([[0.5, -0.2, 0.1],
                    [0.0, 0.3, -0.4]], dtype=torch.float32)
W_h = torch.tensor([[0.2, 0.1],
                    [-0.3, 0.4]], dtype=torch.float32)
b = torch.tensor([0.0, 0.1], dtype=torch.float32)

x = torch.stack([x1, x2, x3, x4]).unsqueeze(0)  # (1, 4, 3)

rnn = nn.RNN(input_size=3, hidden_size=2, batch_first=True)
# Set weights manually
with torch.no_grad():
    rnn.weight_ih_l0.data = W_x
    rnn.weight_hh_l0.data = W_h
    rnn.bias_ih_l0.data = b
    rnn.bias_hh_l0.data = torch.zeros(2, dtype=torch.float32)

output, h_n = rnn(x)

print(f"x.shape: {x.shape}")
print(f"output.shape: {output.shape}")
print(f"h_n.shape: {h_n.shape}")
print(f"output: {output}")
print(f"h_n: {h_n}")

print("1. output[:, -1, :] is the last output, which for RNN is the final hidden state.")
print("2. h_n[-1] is the final hidden state.")
print("3. They are the same for one-layer RNN.")
print("4. If batch_first=False, shapes are (seq_len, batch_size, input_size), output (seq_len, batch_size, hidden), h_n (num_layers, batch_size, hidden).")

x.shape: torch.Size([1, 4, 3])
output.shape: torch.Size([1, 4, 2])
h_n.shape: torch.Size([1, 1, 2])
output: tensor([[[ 0.4621,  0.0997],
         [-0.0973,  0.2924],
         [ 0.1093, -0.1526],
         [ 0.2973,  0.2969]]], grad_fn=<TransposeBackward1>)
h_n: tensor([[[0.2973, 0.2969]]], grad_fn=<StackBackward0>)
1. output[:, -1, :] is the last output, which for RNN is the final hidden state.
2. h_n[-1] is the final hidden state.
3. They are the same for one-layer RNN.
4. If batch_first=False, shapes are (seq_len, batch_size, input_size), output (seq_len, batch_size, hidden), h_n (num_layers, batch_size, hidden).


# Part C: DNA Sequence Classifier

In [13]:
# C1. Generate Data
nucleotides = ['A', 'C', 'G', 'T']
motif = 'ATG'
char2idx = {"A": 0, "C": 1, "G": 2, "T": 3, "<PAD>": 4}

def generate_sequence(length):
    return ''.join(random.choice(nucleotides) for _ in range(length))

def generate_positive(length):
    seq = generate_sequence(length - len(motif))
    pos = random.randint(0, len(seq))
    return seq[:pos] + motif + seq[pos:]

def generate_negative(length):
    seq = generate_sequence(length)
    while motif in seq:
        seq = generate_sequence(length)
    return seq

train_pos = [generate_positive(random.randint(30, 80)) for _ in range(1000)]
train_neg = [generate_negative(random.randint(30, 80)) for _ in range(1000)]
val_pos = [generate_positive(random.randint(30, 80)) for _ in range(250)]
val_neg = [generate_negative(random.randint(30, 80)) for _ in range(250)]
test_pos = [generate_positive(random.randint(30, 80)) for _ in range(250)]
test_neg = [generate_negative(random.randint(30, 80)) for _ in range(250)]

train_data = [(seq, 1) for seq in train_pos] + [(seq, 0) for seq in train_neg]
val_data = [(seq, 1) for seq in val_pos] + [(seq, 0) for seq in val_neg]
test_data = [(seq, 1) for seq in test_pos] + [(seq, 0) for seq in test_neg]

In [14]:
# C2. Encode and Pad
def encode(seq):
    return [char2idx[c] for c in seq]

class DNADataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, label = self.data[idx]
        return encode(seq), len(seq), label

def collate_fn(batch):
    seqs, lengths, labels = zip(*batch)
    max_len = max(lengths)
    padded = [seq + [4] * (max_len - len(seq)) for seq in seqs]
    return torch.tensor(padded), torch.tensor(lengths), torch.tensor(labels)

train_loader = DataLoader(DNADataset(train_data), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(DNADataset(val_data), batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(DNADataset(test_data), batch_size=32, shuffle=False, collate_fn=collate_fn)

In [15]:
# C3. Train Classifier
class DNAClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, h_n = self.rnn(packed)
        last_hidden = h_n[-1]
        return self.fc(last_hidden)

model = DNAClassifier(len(char2idx), 10, 20, 2, 4)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for x, lengths, y in loader:
        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += len(y)
    return total_loss / len(loader), correct / total

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x, lengths)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += len(y)
    return correct / total

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Train Acc {train_acc:.4f}, Val Acc {val_acc:.4f}")

test_acc = evaluate(model, test_loader)
print(f"Test Acc: {test_acc:.4f}")

Epoch 1: Train Loss 0.6983, Train Acc 0.5100, Val Acc 0.5380
Epoch 2: Train Loss 0.6907, Train Acc 0.5165, Val Acc 0.5300
Epoch 3: Train Loss 0.6895, Train Acc 0.5320, Val Acc 0.5420
Epoch 4: Train Loss 0.6866, Train Acc 0.5410, Val Acc 0.5500
Epoch 5: Train Loss 0.6862, Train Acc 0.5530, Val Acc 0.5480
Epoch 6: Train Loss 0.6823, Train Acc 0.5595, Val Acc 0.5440
Epoch 7: Train Loss 0.6809, Train Acc 0.5705, Val Acc 0.5440
Epoch 8: Train Loss 0.6820, Train Acc 0.5655, Val Acc 0.5680
Epoch 9: Train Loss 0.6806, Train Acc 0.5625, Val Acc 0.5640
Epoch 10: Train Loss 0.6778, Train Acc 0.5810, Val Acc 0.5340
Test Acc: 0.5360


In [16]:
# C4. Generalization Tests
test_cases = [
    "ATG" + "A" * 27,  # near beginning
    "A" * 15 + "ATG" + "C" * 15,  # middle
    "G" * 27 + "ATG",  # near end
    "A" * 50 + "ATG" + "T" * 20,  # longer
    "A" * 10 + "ATG" + "A" * 10 + "ATG" + "C" * 10,  # noisy
]

for seq in test_cases:
    encoded = encode(seq)
    length = len(encoded)
    padded = torch.tensor([encoded + [4] * (80 - length)])  # assume max 80
    lengths = torch.tensor([length])
    logits = model(padded, lengths)
    prob = torch.softmax(logits, 1)[0]
    pred = logits.argmax(1).item()
    true = 1 if motif in seq else 0
    print(f"Seq: {seq[:30]}..., True: {true}, Pred: {pred}, Prob: {prob}")

Seq: ATGAAAAAAAAAAAAAAAAAAAAAAAAAAA..., True: 1, Pred: 0, Prob: tensor([0.6842, 0.3158], grad_fn=<SelectBackward0>)
Seq: AAAAAAAAAAAAAAAATGCCCCCCCCCCCC..., True: 1, Pred: 0, Prob: tensor([0.6237, 0.3763], grad_fn=<SelectBackward0>)
Seq: GGGGGGGGGGGGGGGGGGGGGGGGGGGATG..., True: 1, Pred: 1, Prob: tensor([0.3640, 0.6360], grad_fn=<SelectBackward0>)
Seq: AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA..., True: 1, Pred: 1, Prob: tensor([0.4111, 0.5889], grad_fn=<SelectBackward0>)
Seq: AAAAAAAAAAATGAAAAAAAAAAATGCCCC..., True: 1, Pred: 0, Prob: tensor([0.6237, 0.3763], grad_fn=<SelectBackward0>)
